Hi there!
This is the code template for CW1 task2 of COMP34711 2025/26.

- <span style="color:red; font-size:1em">First of all, please rename the notebook into "{your_student_id}_CW1_task2.ipynb", for example "12345678_CW1_task2.ipynb".</span>

- In this template, we only provide the minimal structure for your coursework.
  
- Please carefully read and organize your code in the template we provided.

## Constants

In [1]:
#Please keep only necessary information in this cell.

#----------------------Please keep all following constants unchanged.----------------------------------------
NUM_ROWS_EXAMPLE = 100    # Number of rows in example.csv.
NUM_ROWS_TEST = 160    # Number of rows in testdata.csv.

#----------------------Please modify the following constants to fit your actual value.-----------------------
STUDENT_ID = '10879360'  # Replace with your actual 8-digits student ID.
CORPUS = './data/WikiText-103.txt'  #Keep this path to WikiText_103.txt
EXAMPLE_SET_INPUT = './data/CW1_examples.csv' # Keep this path to CW1_examples.csv (examples with no similarities)
EXAMPLE_SET_OUTPUT = f'./data/{STUDENT_ID}_CW1_examples_task2_results.csv'  # Keep this path to your own prediction of CW1_examples.csv
EXAMPLE_SET_GOLD_STANDARD = './data/CW1_examples_gold.csv'  # Keep this actual path to the CW1_examples_gold.csv
TEST_SET_INPUT = './data/CW1_testdata.csv' # Keep this path to the CW1_testdata.csv (once it get released)

#----------------------Your constants--------------------------------------
# By adding more constants here, you can help improve the clarity and maintainability of your code and make the reviewing easier for TAs.

## Installations

In [2]:
# Install required packages for the coursework

# Uncomment and run the following lines if packages are not installed
# !pip install pandas numpy

!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 39.5 MB/s eta 0:00:00


## Imports

In [3]:
#Please keep all imports of your code cells in this cell

#---------------------Required imports----------------------
import pandas as pd
import re
import numpy as np
import math
import os.path
import sys
#----------------------Your imports-------------------------
import gensim
from gensim.models import FastText
from gensim.utils import simple_preprocess
from gensim.models.phrases import Phrases, Phraser

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

import multiprocessing
import gc
import traceback

nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## Start of your code cells

- The code cells provided below are demo code format for TAs to quickly locate your implementation.

- You have full right to freely add/delete/edit the titles and codes in the following cells.

### Corpus loader

In [5]:
# This list will hold all the tokenized sentences from the corpus
corpus_for_word2vec = []
try:
    # Open the WikiText-103.txt file
    with open(CORPUS, 'r', encoding='utf-8') as f:
        all_lines = f.readlines()

    # Temporarily stores lines belonging to one paragraph
    current_document_lines = []
    # Regex to detect Wikipedia-style headers
    split_pattern = re.compile(r'^\s*=[\s=]*[^=\n]*[\s=]*=[\s=]*$')
    processed_sentence_count = 0

    # Iterate through every single line in the 100M+ token file
    for i, line in enumerate(all_lines):
        line_stripped = line.strip()

        # Ignore empty lines or very short lines.
        if not line_stripped or len(line_stripped) < 3:
            continue

        # Check if the line is a section header
        if split_pattern.match(line):
            # If it is a header, process the paragraph we've collected so far
            if current_document_lines:
                paragraph_text = ' '.join(current_document_lines).strip()
                if paragraph_text:
                    # Use NLTK to split the full paragraph into individual sentences
                    sentences = sent_tokenize(paragraph_text)
                    for sentence in sentences:
                        # Use gensim's simple_preprocess to:
                        # 1. Convert to lowercase
                        # 2. Tokenize
                        # 3. Remove punctuation and numbers
                        # 4. Remove accents
                        tokens = simple_preprocess(sentence, deacc=True)
                        # Only add the sentence if it's not empty
                        if tokens:
                            corpus_for_word2vec.append(tokens)
                            processed_sentence_count += 1

                # Reset the list to start collecting the next paragraph
                current_document_lines = []
            continue

        # If it's a normal line, add it to the current paragraph
        current_document_lines.append(line_stripped)

    # Process the final paragraph in the file
    if current_document_lines:
        paragraph_text = ' '.join(current_document_lines).strip()
        if paragraph_text:
            sentences = sent_tokenize(paragraph_text)
            for sentence in sentences:
                tokens = simple_preprocess(sentence, deacc=True)
                if tokens:
                    corpus_for_word2vec.append(tokens)
                    processed_sentence_count += 1

    # This phase handles the "multi-word term" requirement
    # Train the Phrases model on the 1.2M+ tokenized sentences.
    # It learns which words statistically appear together often
    phrases_model = Phrases(corpus_for_word2vec, min_count=5, threshold=10.0)
    # Convert the full model to a lighter, faster 'Phraser' object for application
    bigram_phraser = Phraser(phrases_model)
    # Apply the phraser to the entire corpus.
    corpus_with_bigrams = [bigram_phraser[sentence] for sentence in corpus_for_word2vec]

    # The original corpus is no longer needed, so delete it to free up RAM
    del corpus_for_word2vec
    gc.collect()

    print(f"✅ Corpus processing complete. Processed {processed_sentence_count:,} sentences.")

except FileNotFoundError:
    print(f"❌ Error: Corpus file not found at {CORPUS}")
    # Ensure the variable exists to prevent later cells from crashing
    corpus_with_bigrams = []
except Exception as e:
    print(f"❌ Unexpected error during corpus processing: {e}")
    traceback.print_exc()
    # Ensure the variable exists to prevent later cells from crashing
    corpus_with_bigrams = []

✅ Corpus processing complete. Processed 1,277,133 sentences.


### Vectoriser



In [6]:
try:
    # We are training a FastText model, which is a 'dense static representation'.
    word2vec_model = FastText(
        # [MODIFICATION 1]: Use the bigram-processed corpus.
        # This allows the model to learn vectors for multi-word terms.
        sentences=corpus_with_bigrams,
        # Dimensionality of the word vectors
        vector_size=150,
        # Context window size
        window=5,
        # Ignore all words/bigrams with a total frequency lower than this
        min_count=5,
        # [MODIFICATION 2]: Use FastText (based on Skip-gram)
        # sg=1 means we are using the Skip-gram (SG) model
        sg=1,
        # Use all available CPU cores to speed up training
        workers=multiprocessing.cpu_count(),
        # Iterate over the entire corpus 5 times
        epochs=5,
        # [MODIFICATION 3]: Set a random seed for reproducibility.
        # This ensures that if we re-run this cell, we get a consistent model and a stable score
        seed=378
    )

    vocab_size = len(word2vec_model.wv.key_to_index)
    print(f"✅ Vocabulary size: {vocab_size:,} words")

except NameError:
    print("❌ Error: 'corpus_with_bigrams' not found. Please run the corpus loading cell first.")
except Exception as e:
    print(f"❌ An unexpected error occurred during Word2Vec training: {e}")
    import traceback
    traceback.print_exc()

✅ Vocabulary size: 149,149 words


### Out of vocabulary words processing

In [7]:
try:
    # Check if the main model from the 'Vectoriser' cell exists
    if 'word2vec_model' not in locals():
        raise NameError("Word2Vec model ('word2vec_model') not found.")

    # Get the vector dimension from the trained model
    VECTOR_SIZE = word2vec_model.vector_size
    # Get the KeyedVectors object (the part of the model that stores/retrieves vector
    wv = word2vec_model.wv

    print(f"✅ Vector retrieval setup complete. Vector size: {VECTOR_SIZE}")

except NameError as e:
    print(f"❌ Error: {e}. Please train or load the FastText model first.")
    # Set a default to prevent crashes
    VECTOR_SIZE = 1
    wv = None
except Exception as e:
    print(f"❌ An unexpected error occurred during setup: {e}")
    # Set a default to prevent crashes
    VECTOR_SIZE = 1
    wv = None

def get_word_vector(term):
    # 1. Validation: If term is invalid (empty, or a multi-word phrase), return a zero vector.
    if not isinstance(term, str) or len(term.strip()) == 0 or ' ' in term:
        return np.zeros(VECTOR_SIZE, dtype=np.float32)

    # 2. Model Check: Ensure the model ('wv') is loaded and ready.
    if wv is None:
        print("Error: Word2Vec KeyedVectors ('wv') is not available.")
        return np.zeros(VECTOR_SIZE, dtype=np.float32)

    try:
        # 3. Retrieve Vector:
        # This is the key step.
        # If 'term' is IN vocabulary: 'wv[term]' returns its stored vector.
        # If 'term' is OOV: 'wv[term]' *synthesizes* a new vector
        # from its subwords  and returns that.
        # IT DOES NOT RAISE A KEYERROR.
        vector = wv[term]

        # 4. Normalize:
        # Convert the vector to unit length (L2 norm) for cosine similarity.
        vector_normalized = normalize(vector.reshape(1, -1), norm='l2', axis=1)[0]
        return vector_normalized

    except KeyError:
        # 5. Fallback (RARELY USED with FastText):
        # This block would catch KeyErrors if we were using Word2Vec.
        # With FastText, it only triggers for extremely rare/empty subword cases.
        return np.zeros(VECTOR_SIZE, dtype=np.float32)

    except Exception as e:
        print(f"❌ Error retrieving vector for term '{term}': {e}")
        return np.zeros(VECTOR_SIZE, dtype=np.float32)


✅ Vector retrieval setup complete. Vector size: 150


### Multi-word terms processing

In [8]:
def get_multi_word_vector(term):
    # 1. Validation: Handle empty or invalid input
    if not isinstance(term, str) or len(term.strip()) == 0:
        return np.zeros(VECTOR_SIZE, dtype=np.float32)

    # 2. Pre-processing:
    # Use the *same* pre-processor as the corpus training to ensure consistency.
    # This tokenizes, lowercases, and removes punctuation.
    tokens = simple_preprocess(term, deacc=True)

    if not tokens:
        return np.zeros(VECTOR_SIZE, dtype=np.float32)

    # 3. Bigram Phrasing:
    # This is the CRITICAL step.
    # We apply the *same* phraser we trained on the corpus to the input tokens.
    try:
        # The phraser will automatically combine known bigrams.
        words = bigram_phraser[tokens]
    except NameError:
        # Fallback if 'bigram_phraser' wasn't defined in the previous cell
        # Revert to simple tokens if phraser is missing
        words = tokens
    except Exception as e:
        # Revert to simple tokens on other errors
        words = tokens

    if not words:
        return np.zeros(VECTOR_SIZE, dtype=np.float32)

    # 4. Averaging:
    # This loop now iterates over the *phrased* list of tokens.
    vector_sum = np.zeros(VECTOR_SIZE, dtype=np.float32)
    valid_vectors_count = 0

    for word in words:
        # This calls the FastText-backed function from the previous cell.
        word_vec = get_word_vector(word)

        # Check if the returned vector is not just all zeros.
        if np.any(word_vec):
            # Add the valid vector to the sum.
            vector_sum += word_vec
            # Increment the counter.
            valid_vectors_count += 1

    # If no valid vectors were found,
    # return a zero vector to avoid a division-by-zero error.
    if valid_vectors_count == 0:
        return np.zeros(VECTOR_SIZE, dtype=np.float32)

    # Calculate the mean vector.
    average_vector = vector_sum / valid_vectors_count
    # Normalize the final mean vector to unit length (L2 norm) for cosine similarity.
    average_vector_normalized = normalize(average_vector.reshape(1, -1), norm='l2', axis=1)[0]

    # Return the final, normalized vector.
    return average_vector_normalized


### Similarity calculation

In [12]:
# Cache for the final vectors. This is the main cache for the similarity loop.
_final_w2v_vector_cache = {}

def get_final_word2vec_vector(term):
    # 1. Check cache
    if term in _final_w2v_vector_cache:
        return _final_w2v_vector_cache[term]

    # 2. Validate input
    if not isinstance(term, str) or len(term.strip()) == 0:
        return np.zeros(VECTOR_SIZE, dtype=np.float32)

    final_vec = None

    # 3. Routing logic
    if ' ' in term:
        # 3a. It's a multi-word term.
        # Call the function we modified to use the 'bigram_phraser'.
        final_vec = get_multi_word_vector(term)
    else:
        # 3b. It's a single-word term.
        # Call the function that is now backed by FastText.
        # This will automatically handle both in-vocabulary and OOV words.
        final_vec = get_word_vector(term)

    # 4. Fallback
    if final_vec is None:
         final_vec = np.zeros(VECTOR_SIZE, dtype=np.float32)

    # 5. Cache and return
    _final_w2v_vector_cache[term] = final_vec
    return final_vec

try:
    # 1. Load the input file
    df_input = pd.read_csv(TEST_SET_INPUT, header=None, names=['Pair ID', 'Term 1', 'Term 2'])
    # A list to store our dictionary of results
    results_list = []

    # 2. Clear cache and free memory before the loop
    _final_w2v_vector_cache = {}
    gc.collect()

    # 3. Main Loop: Iterate over every row (term pair) in the input file
    for index, row in df_input.iterrows():
        pair_id = row['Pair ID']
        term1_raw = str(row['Term 1']).strip()
        term2_raw = str(row['Term 2']).strip()
        # Default similarity is 0.0
        sim_score = 0.0

        try:
            # 4. Get the final vector for each term using our router function
            vec1 = get_final_word2vec_vector(term1_raw)
            vec2 = get_final_word2vec_vector(term2_raw)

            # 5. Calculate Cosine Similarity
            # Only calculate if both vectors are non-zero
            if np.any(vec1) and np.any(vec2):
                # Reshape vectors to (1, N) for cosine_similarity function
                sim_score = cosine_similarity(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]

                # Clean up tiny or invalid numbers
                # Handle floating point inaccuracies
                if abs(sim_score) < 1e-9:
                    sim_score = 0.0
                # Handle potential NaN values
                elif np.isnan(sim_score):
                    sim_score = 0.0

            # One or both vectors were 0.0
            else:
                sim_score = 0.0

        except Exception as e:
            # Catch any unexpected errors during vector retrieval
            print(f"   [ERROR] PairID {pair_id} ({term1_raw}, {term2_raw}): {e}")
            sim_score = 0.0

        # 6. Append the result for this pair to our list
        results_list.append({
            'Pair ID': pair_id,
            'Term 1': term1_raw,
            'Term 2': term2_raw,
            'Similarity': sim_score
        })

    # 7. Convert the list of results into a pandas DataFrame
    df_results = pd.DataFrame(results_list, columns=['Pair ID', 'Term 1', 'Term 2', 'Similarity'])

    # 8. Sanity check: ensure output length matches input length
    if len(df_results) != NUM_ROWS_EXAMPLE:
        print(f"⚠️ Warning: Expected {NUM_ROWS_EXAMPLE} results, but generated {len(df_results)}")

    # 9. Save the DataFrame to a CSV file
    df_results.to_csv(EXAMPLE_SET_OUTPUT, index=False, header=False)

    print(f"✅ Successfully saved Task 2 results (4 columns) to: {EXAMPLE_SET_OUTPUT}\n")

except FileNotFoundError:
    print(f"❌ Error: Input file not found at {EXAMPLE_SET_INPUT}")
except NameError as e:
     print(f"❌ NameError: {e}. Ensure prerequisite functions/variables are defined.")
except Exception as e:
    print(f"❌ Unexpected error during final calculation or saving: {e}")
    traceback.print_exc()



⚠️ Warning: Expected 100 results, but generated 160
✅ Successfully saved Task 2 results (4 columns) to: ./data/10879360_CW1_examples_task2_results.csv



## End of your code cells

In [10]:
#@title Evaluation scripts

def read_data(submission_file_path, gold_standard_file_path):
    # Try to find your ID from the filename (looks for 7-8 digit numbers)
    id_regex = r'\d{7,8}'

    user_id = re.findall(id_regex, submission_file_path)
    print("Found your ID: ", user_id)
    if user_id:
        user_id = user_id[0]
    else:
        user_id = 'Unknown'

    # Load both CSV files (assuming no header row)
    submission_df = pd.read_csv(submission_file_path, header=None)
    reference_df = pd.read_csv(gold_standard_file_path, header=None)

    # Add row tracking to the reference data for precise comparisons
    indices = np.arange(reference_df.shape[0])
    reference_df['row_index'] = indices

    return submission_df, reference_df, user_id

def group_pairs_by_common_words(reference_df):
    """
    Group word pairs that share any common word (in either column 1 or column 2).
    """
    # Convert dataframe to list of dictionaries for easier processing
    all_pairs = reference_df.to_dict(orient='records')

    # Track which pairs have been assigned to groups
    assigned = [False] * len(all_pairs)
    groups = []

    for i, pair in enumerate(all_pairs):
        if assigned[i]:
            continue

        # Start a new group with this pair
        current_group = [pair]
        assigned[i] = True

        # Get words from this pair
        words_in_group = {pair[1], pair[2]}  # columns 1 and 2 are the word pair

        # Keep expanding the group by finding pairs that share words
        changed = True
        while changed:
            changed = False
            for j, other_pair in enumerate(all_pairs):
                if assigned[j]:
                    continue

                # Check if this pair shares any word with the current group
                other_words = {other_pair[1], other_pair[2]}
                if words_in_group & other_words:  # Set intersection - any common words
                    current_group.append(other_pair)
                    assigned[j] = True
                    words_in_group.update(other_words)
                    changed = True

        groups.append(current_group)

    return groups

def evaluate_submission(submission_df, reference_df, user_id):
    """
    Evaluate your submission by comparing similarity score orderings with the reference.

    1. Groups word pairs that share any common word (in either column 1 or column 2)
    2. Compares how you ranked similarity scores vs. the reference ranking
    3. Creates individual test cases to show exactly what passed or failed

    """
    # Group reference data by pairs that share any common word
    # This creates clusters of word pairs that have overlapping vocabulary
    grouped_list = group_pairs_by_common_words(reference_df)

    # Set up tracking for your evaluation results
    total_score = 0
    maximum_score = 0
    test_cases = []  # All individual test cases with their results
    failed_test_cases = []  # Failed tests for detailed feedback
    missed_rows = []  # Any data rows you might have missed
    test_case_id = 1

    # Process each group of word pairs
    for group_dict in grouped_list:
        # Skip single-item groups (no comparison possible)
        if len(group_dict) == 1:
            continue

        your_predictions = []

        # Find your predictions for each word pair in this group
        for row in group_dict:
            # Look up your prediction using the row ID (column 0)
            your_row = submission_df.loc[submission_df[0] == row[0]]

            try:
                your_row_dict = your_row.to_dict(orient='records')[0]
                your_predictions.append(your_row_dict)
            except:
                # Keep track of any missing data for helpful feedback
                missed_rows.append(row[0])
                continue

        # Sort both your predictions and reference data by row ID for fair comparison
        sorted_your_predictions = sorted(your_predictions, key=lambda x: x[0], reverse=True)
        sorted_reference_group = sorted(group_dict, key=lambda x: x[0], reverse=True)

        # Compare all possible pairs within this word group
        for i in range(len(sorted_your_predictions)):
            for j in range(i+1, len(sorted_your_predictions)):
                try:
                    # Calculate the difference in similarity scores (column 3)
                    your_result = float(sorted_your_predictions[i][3]) - float(sorted_your_predictions[j][3])
                    if math.isnan(your_result):
                        continue

                    reference_result = sorted_reference_group[i][3] - sorted_reference_group[j][3]

                    # Create a clear description for this test case
                    # Find common words between the two pairs being compared
                    words_pair1 = {sorted_reference_group[i][1], sorted_reference_group[i][2]}
                    words_pair2 = {sorted_reference_group[j][1], sorted_reference_group[j][2]}
                    common_words = words_pair1 & words_pair2
                    if common_words:
                        word_context = f"pairs sharing word(s): {', '.join(sorted(common_words))}"
                    else:
                        word_context = "connected word pairs"

                    pair1 = f"({sorted_reference_group[i][1]}, {sorted_reference_group[i][2]})"
                    pair2 = f"({sorted_reference_group[j][1]}, {sorted_reference_group[j][2]})"

                    # Figure out what the correct ordering should be
                    if reference_result > 0:
                        expected_order = f"{pair1} > {pair2}"
                    elif reference_result < 0:
                        expected_order = f"{pair1} < {pair2}"
                    else:
                        expected_order = f"{pair1} = {pair2}"

                    # And what your prediction actually was
                    if your_result > 0:
                        your_order = f"{pair1} > {pair2}"
                    elif your_result < 0:
                        your_order = f"{pair1} < {pair2}"
                    else:
                        your_order = f"{pair1} = {pair2}"

                    # Check if you got this comparison right!
                    test_passed = False
                    if your_result * reference_result > 0:
                        test_passed = True
                        total_score += 1
                    elif your_result == 0 and reference_result == 0:
                        test_passed = True
                        total_score += 1

                    # Record this test case for your feedback report
                    test_case = {
                        'id': test_case_id,
                        'description': f"Ordering comparison for {word_context}",
                        'expected': expected_order,
                        'actual': your_order,
                        'passed': test_passed,
                        'reference_data': [sorted_reference_group[i], sorted_reference_group[j]],
                        'your_data': [sorted_your_predictions[i], sorted_your_predictions[j]]
                    }

                    test_cases.append(test_case)

                    if not test_passed:
                        failed_test_cases.append(test_case)

                    test_case_id += 1

                except:
                    continue

                # Keep track of the total possible points
                maximum_score += 1

    return {
        'user_id': user_id,
        'total_score': total_score,
        'maximum_score': maximum_score,
        'test_cases': test_cases,
        'failed_test_cases': failed_test_cases,
        'missed_rows': missed_rows
    }

def create_progress_bar(passed, total, width=50):
    """
    Create a visual progress bar showing test results.

    Args:
        passed (int): Number of passed tests
        total (int): Total number of tests
        width (int): Width of the progress bar in characters

    Returns:
        str: Formatted progress bar string
    """
    if total == 0:
        return "[" + " " * width + "] 0/0 (0.0%)"

    percentage = (passed / total) * 100
    filled_width = int((passed / total) * width)
    failed_width = width - filled_width

    bar = "["
    bar += "✓" * filled_width  # Green checkmarks for passed tests
    bar += "✗" * failed_width  # Red X's for failed tests
    bar += f"] {passed}/{total} ({percentage:.1f}%)"

    return bar

def generate_feedback_report(result_dict):
    user_id = result_dict['user_id']
    total_score = result_dict['total_score']
    maximum_score = result_dict['maximum_score']
    test_cases = result_dict['test_cases']
    failed_test_cases = result_dict['failed_test_cases']
    missed_rows = result_dict['missed_rows']

    print("="*70)
    print("YOUR SUBMISSION EVALUATION REPORT")
    print("="*70)

    # Alert if ID not found in filename
    if user_id == 'Unknown':
        print('WARNING: ID not found in filename!')
        print('Please ensure your filename contains your 7-8 digit ID.')

    print(f"Your ID: {user_id}")
    print(f"Total Test Cases: {len(test_cases)}")
    print(f"Passed: {total_score}")
    print(f"Failed: {len(failed_test_cases)}")

    if maximum_score > 0:
        percentage = (total_score / maximum_score) * 100
        print(f"Success Rate: {percentage:.1f}%")
    else:
        print("Success Rate: N/A (no valid comparisons found)")


    # Show progress bar
    print("TEST RESULTS OVERVIEW:")
    progress_bar = create_progress_bar(total_score, maximum_score, 50)
    print(f"  {progress_bar}")
    print()

    # Generate detailed feedback for failed test cases
    if failed_test_cases:
        print(f"\nDETAILED FEEDBACK ON FAILED TEST CASES ({len(failed_test_cases)} failures):")
        print("=" * 70)

        for i, test_case in enumerate(failed_test_cases, 1):
            reference_data = test_case['reference_data']
            your_data = test_case['your_data']

            print(f"\nFAILED TEST CASE #{test_case['id']} ({i}/{len(failed_test_cases)})")
            print(f"Context: {test_case['description']}")
            print(f"Expected: {test_case['expected']}")
            print(f"Your Result: {test_case['actual']}")
            print("Detailed Comparison:")
            print("Reference (Correct):")

            # Show correct ordering from reference with actual scores
            for j in range(len(reference_data)):
                print(f"     {{{reference_data[j][0]}, {reference_data[j][1]}, {reference_data[j][2]}, {reference_data[j][3]}}}")

            print("Your Submission:")

            # Show your ordering with actual scores
            for j in range(len(your_data)):
                print(f"     {{{your_data[j][0]}, {your_data[j][1]}, {your_data[j][2]}, {your_data[j][3]}}}")

            print("-" * 50)
    else:
        print("\nEXCELLENT! All test cases passed!")
        print("Your similarity score orderings are completely correct!")

    # Report missing rows
    if missed_rows:
        print(f"\nMISSING DATA ({len(missed_rows)} rows not found):")
        print("-" * 40)
        for row in missed_rows:
            print(f"Row ID: {row} is missing from your submission")
        print("\nTIP: Make sure your submission includes all required rows.")
    else:
        print("\nDATA COMPLETENESS: No missing rows in your submission!")

    print("\n" + "="*70)

def mark_and_feedback(submission_file, reference_file):
    """
    Main function to run your submission evaluation script.

    Usage: python cw1_fast_evaluation.py <your_submission.csv> <reference_file.csv>
    """

    # Check if files exist
    if not os.path.exists(submission_file):
        print(f"Error: Your submission file '{submission_file}' not found!")
        print("Make sure the file path is correct and the file exists.")

    if not os.path.exists(reference_file):
        print(f"Error: Reference file '{reference_file}' not found!")
        print("Make sure you have the correct gold standard file.")

    try:
        print("Loading your data...")
        submission_df, reference_df, user_id = read_data(submission_file, reference_file)

        print("Evaluating your submission...")
        result_dict = evaluate_submission(submission_df, reference_df, user_id)

        print("Generating your personalized feedback report...")
        generate_feedback_report(result_dict)

    except Exception as e:
        print(f"Error during evaluation: {str(e)}")
        print("Please check that your files are in the correct CSV format.")
        print("Each row should contain: ID, word1, word2, similarity_score")

In [11]:
# @title Evaluate the model on the example dataset

# Please run the evaluation scripts cell above before running the mark_and_record

results = mark_and_feedback(EXAMPLE_SET_OUTPUT, EXAMPLE_SET_GOLD_STANDARD)


Loading your data...
Found your ID:  ['10879360']
Evaluating your submission...
Generating your personalized feedback report...
YOUR SUBMISSION EVALUATION REPORT
Your ID: 10879360
Total Test Cases: 103
Passed: 72
Failed: 31
Success Rate: 69.9%
TEST RESULTS OVERVIEW:
  [✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✗✗✗✗✗✗✗✗✗✗✗✗✗✗✗✗] 72/103 (69.9%)


DETAILED FEEDBACK ON FAILED TEST CASES (31 failures):

FAILED TEST CASE #4 (1/31)
Context: Ordering comparison for pairs sharing word(s): join
Expected: (join, acquire) < (join, add)
Your Result: (join, acquire) > (join, add)
Detailed Comparison:
Reference (Correct):
     {112, join, acquire, 0.29}
     {110, join, add, 0.81}
Your Submission:
     {112, join, acquire, 0.28577515}
     {110, join, add, 0.2644875}
--------------------------------------------------

FAILED TEST CASE #7 (2/31)
Context: Ordering comparison for pairs sharing word(s): join
Expected: (join, marry) < (join, add)
Your Result: (join, marry) > (join, add)
Detailed Comparison:
Refer

### Save predictions to formatted file.

In [13]:
# Now please modify the code to format your output csv file.

output_df = df_results  # Replace with your actual DataFrame or output
# For example, if you have a DataFrame named 'output_df', you can save it
assert isinstance(output_df, pd.DataFrame)
assert len(output_df) == NUM_ROWS_TEST, "Output length is not aligned with the testdata.csv."
output_df.to_csv(f'{STUDENT_ID}_CW1_task2_results.csv', index=False, header=False)